# Convolutional Neural Networks for Image Classification with PyTorch

## Overview

In this notebook, we build and train a convolutional neural network (CNN) for handwritten digit classification using the MNIST dataset.

The notebook introduces several fundamental concepts in modern deep learning, including:

* image preprocessing,
* convolutional neural networks,
* regularization,
* validation-based model selection,
* and classification evaluation metrics.

Unlike traditional machine learning models that rely on manually engineered features, CNNs automatically learn visual representations directly from image data.

The experiment demonstrates how deep neural networks:

* extract hierarchical visual features,
* improve classification performance,
* and generalize to unseen data.

---

## Learning Goals

The notebook is designed to introduce several core ideas in computer vision and deep learning:

* how convolutional layers learn spatial patterns,
* why normalization improves optimization,
* how early stopping reduces overfitting,
* how validation sets guide model selection,
* and how classification metrics evaluate model quality.

The workflow also reinforces best practices such as:

* reproducibility,
* train/validation/test separation,
* and proper evaluation procedures.

---

## Pipeline Summary

The experiment follows a complete supervised learning workflow:

1. Import required libraries and configure reproducibility.
2. Load and preprocess the MNIST dataset.
3. Create train, validation, and test splits.
4. Visualize the dataset and inspect class balance.
5. Define a convolutional neural network architecture.
6. Train the model using mini-batch gradient descent.
7. Apply early stopping based on validation performance.
8. Monitor learning curves during optimization.
9. Evaluate the final model on the test set.
10. Analyze predictions using confusion matrices and classification reports.
11. Save pretrained convolutional features for future transfer learning experiments.

---

## About the MNIST Dataset

MNIST is one of the most widely used benchmark datasets in machine learning.

It contains:

* grayscale images of handwritten digits,
* image size of 28×28 pixels,
* and 10 target classes representing digits from 0 to 9.

The dataset is intentionally simple, making it ideal for learning:

* neural network training,
* optimization,
* and computer vision fundamentals.

Despite its simplicity, MNIST demonstrates many concepts that generalize to larger real-world image recognition problems.

---

## Key Deep Learning Concepts

This notebook demonstrates several important ideas in deep learning:

### Convolutional Feature Extraction

CNNs learn local spatial patterns such as:

* edges,
* curves,
* corners,
* and shapes.

These low-level features combine into more complex visual representations deeper in the network.

### Regularization and Generalization

Deep networks can memorize training data if trained too aggressively.

We therefore apply:

* dropout,
* weight decay,
* and early stopping

to improve generalization performance.

### Validation-Based Model Selection

The validation set is used to:

* monitor overfitting,
* tune training duration,
* and select the best model checkpoint.

The test set remains untouched until the very end.

### Classification Metrics

Model quality is evaluated using:

* accuracy,
* F1 scores,
* confusion matrices,
* and class-wise precision/recall statistics.

These metrics provide a more complete view of performance than accuracy alone.

---

## Output

The notebook generates:

* training and validation learning curves,
* classification accuracy metrics,
* macro and weighted F1 scores,
* confusion matrices,
* class-wise evaluation reports,
* and pretrained convolutional feature weights.

The final result is a fully trained CNN capable of classifying handwritten digits with high accuracy.


# Imports and Environment Setup

This notebook uses:
- PyTorch for deep learning and GPU acceleration,
- torchvision for datasets and image preprocessing,
- scikit-learn for evaluation metrics,
- and matplotlib for visualization.

Additional utilities are used for:
- reproducibility,
- dataset inspection,
- and training progress monitoring.


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import multiprocessing
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import MNIST

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)


# Reproducibility and Device Selection

Deep learning experiments contain several sources of randomness, including:
- parameter initialization,
- mini-batch shuffling,
- and GPU kernel behavior.

To improve reproducibility, we fix the random seeds for:
- Python,
- NumPy,
- and PyTorch.

The notebook also automatically selects the best available compute device:
- CUDA for NVIDIA GPUs,
- MPS for Apple Silicon GPUs,
- or CPU fallback.

In [ ]:
def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


In [ ]:
seed_everything()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")


# Image Preprocessing

Neural networks operate on tensors rather than raw images, so several preprocessing steps are applied:
- resizing images to a fixed spatial resolution,
- converting images to tensors,
- and normalizing pixel intensities.

MNIST normalization uses the dataset mean and standard deviation:

$x_{\text{norm}} = \frac{x - \mu}{\sigma}$

Normalization improves optimization stability and helps gradient-based training converge more reliably.


In [ ]:
mnist_transforms = transforms.Compose([

    transforms.Resize((28, 28)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.1307,),
        std=(0.3081,)
    )
])


# Loading the MNIST Dataset

MNIST is a standard handwritten digit classification dataset containing:
- grayscale images,
- 10 digit classes (0–9),
- 60,000 training samples,
- and 10,000 test samples.

The original training set is further partitioned into:
- a training split used for optimization,
- and a validation split used for model selection and early stopping.

The test set remains untouched until final evaluation.


In [ ]:
# 1. Load the complete training dataset (60,000 images)
mnist_full_train = MNIST(
    root="data",
    # root="/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets",
    train=True,
    download=True,
    transform=mnist_transforms
)

# 2. Partition into train (80% / 48,000) and validation (20% / 12,000)
# Passing a manual seed generator guarantees the exact same split across runs
generator = torch.Generator().manual_seed(42)
mnist_train_dataset, mnist_val_dataset = random_split(
    mnist_full_train, 
    [0.8, 0.2], 
    generator=generator
)

# 3. Load the test dataset (remains untouched, 10,000 images)
mnist_test_dataset = MNIST(
    root="data",
    # root="/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets",
    train=False,
    download=True,
    transform=mnist_transforms
)

# DataLoaders

PyTorch DataLoaders provide efficient mini-batch iteration during training.

Mini-batch training:
- reduces memory usage,
- improves optimization efficiency,
- and produces more stable gradient estimates.

The training loader uses random shuffling to avoid learning artifacts from sample ordering.

In [ ]:
batch_size = 128

pin_memory = (device.type == "cuda")
num_workers = min(2, multiprocessing.cpu_count())
persistent_workers = (num_workers > 0)

mnist_train_loader = DataLoader(
    mnist_train_dataset,
    batch_size=batch_size,
    shuffle=True, num_workers=num_workers,
    pin_memory=pin_memory, persistent_workers=persistent_workers
)

mnist_val_loader = DataLoader(
    mnist_val_dataset,
    batch_size=batch_size,
    shuffle=False, num_workers=num_workers,
    pin_memory=pin_memory, persistent_workers=persistent_workers
)


mnist_test_loader = DataLoader(
    mnist_test_dataset,
    batch_size=batch_size,
    shuffle=False, num_workers=num_workers,
    pin_memory=pin_memory, persistent_workers=persistent_workers
)


# Dataset Inspection

Before training, it is useful to inspect the dataset visually and verify:
- image preprocessing,
- label correctness,
- and class balance.

This step also helps identify potential issues such as:
- corrupted samples,
- incorrect normalization,
- or class imbalance.


In [ ]:
def visualize_dataset(dataset, num_samples=8):
    mean = np.array([0.1307])
    std = np.array([0.3081])
    fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 2, 3))
    total_samples = len(dataset)
    random_indices = random.sample(range(total_samples), num_samples)
    if hasattr(dataset, "dataset"):
        classes = dataset.dataset.classes
    else:
        classes = dataset.classes
    for i, idx in enumerate(random_indices):
        image_tensor, label_idx = dataset[idx]
        img = image_tensor.squeeze().numpy()
        img = std * img + mean
        img = np.clip(img, 0, 1)
        class_letter = classes[label_idx]
        # 2. Add cmap="gray" to override the default yellow/purple style
        axes[i].imshow(img, cmap="gray")
        # -----------------
        axes[i].set_title(f"Label: {class_letter}", fontsize=11, fontweight='bold')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# Call it directly on your subset dataset
visualize_dataset(mnist_train_dataset, num_samples=8)


# Class Distribution Analysis

We compute the number of samples for each digit class to verify dataset balance.

Balanced datasets generally simplify optimization because the model receives comparable training signal across classes.

In [ ]:
# Count samples per class
class_names = [str(i) for i in range(10)]

# Instantly extracts labels in milliseconds without reading or transforming images
train_indices = mnist_train_dataset.indices
train_labels = mnist_full_train.targets[train_indices].tolist()
class_counts = Counter(train_labels)

print("Number of images per class:\n")
for i, count in class_counts.items():
    print(f"{class_names[i]:30s}: {count}")


In [ ]:
counts = [class_counts[i] for i in range(len(class_names))]

plt.figure(figsize=(8,4))
plt.bar(range(len(class_names)), counts)
plt.xlabel("Class index")
plt.ylabel("Number of images")
plt.title("Dataset class distribution")
plt.show()


# Convolutional Neural Network Architecture

We define a compact convolutional neural network for digit classification.

The architecture is divided into two components:

## Feature Extractor

The convolutional layers learn hierarchical visual representations such as:
- edges,
- strokes,
- corners,
- and higher-level digit structures.

Pooling layers progressively reduce spatial dimensionality while preserving important features.

## Classifier

The fully connected layers map learned visual representations to class probabilities.

This separation between:
- feature extraction,
- and classification

is especially important in transfer learning workflows.


## Visualizing the Spatial Transformations

To help your students intuitively track how spatial resolution shrinks while feature depth expands, here is a visual reference you can drop directly into your architectural summary markdown section:

| Pipeline Stage | Layer Type | Output Activation Shape ($C \times H \times W$) | Rationale |
| --- | --- | --- | --- |
| **Input** | Raw Preprocessed Image | $1 \times 28 \times 28$ | Single-channel grayscale MNIST sample. |
| **Stage 1 (Conv)** | `nn.Conv2d(1, 32, k=3, p=1)` | $32 \times 28 \times 28$ | Padding preserves spatial boundaries; channels expand to 32. |
| **Stage 1 (Pool)** | `nn.MaxPool2d(2)` | $32 \times 14 \times 14$ | $2\times2$ pooling downsamples spatial height and width by half. |
| **Stage 2 (Conv)** | `nn.Conv2d(32, 64, k=3, p=1)` | $64 \times 14 \times 14$ | Features deepen to collect more abstract structural shapes. |
| **Stage 2 (Pool)** | `nn.MaxPool2d(2)` | $64 \times 7 \times 7$ | Final spatial reduction. Features are now ready for flattening. |
| **Classifier** | `nn.Flatten()` | $3136$ vector elements | Total inputs passed directly to the first dense layer ($64 \times 7 \times 7$). |


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            # Regularizes dense classifier layers
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Early Stopping

As training progresses, neural networks may begin to overfit the training data.

Typical overfitting behavior includes:
- decreasing training loss,
- stagnant or worsening validation performance,
- and reduced generalization.

Early stopping is a regularization strategy that:
- monitors validation performance,
- saves the best-performing model,
- and terminates training once improvement stalls.

This prevents unnecessary optimization after generalization performance has peaked.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.best_acc = -float("inf")
        self.best_weights = None
        self.best_epoch = 0
        self.min_delta = min_delta 

    def step(self, model, val_loss, val_acc, epoch):

        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_acc = val_acc
            self.best_epoch = epoch+1
            self.counter = 0
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False

        self.counter += 1
        print(
            f"[EarlyStopping] Epoch {epoch+1 if epoch is not None else ''}: "
            f"No improvement → counter {self.counter}/{self.patience}"
        )
        return self.counter >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_weights)


# Evaluation Function

We define a reusable evaluation routine for validation and test inference.

During evaluation:
- gradients are disabled using `torch.no_grad()`,
- and the model is switched to evaluation mode using `model.eval()`.

This ensures:
- deterministic inference behavior,
- lower memory usage,
- and correct BatchNorm / Dropout behavior.


In [ ]:
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item() * x.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == y).sum().item()
            total += y.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


# Training on MNIST

The CNN is trained using supervised learning on the MNIST training split.

During optimization:
- convolutional filters learn reusable visual patterns,
- classifier layers learn class-specific decision boundaries,
- and validation performance is monitored after each epoch.

Optimization uses:
- Cross Entropy Loss for multi-class classification,
- AdamW for adaptive gradient optimization with decoupled weight decay,
- and Early Stopping for regularization.


In [ ]:
model = SimpleCNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-2
)

early_stopping = EarlyStopping(patience=10)

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

print("TRAINING CNN ON MNIST")

epochs = 100

for epoch in range(epochs):

    # Training
    model.train()

    total_train_loss = 0
    correct = 0
    total = 0

    for x, y in tqdm(mnist_train_loader):
        
        x = x.to(device)
        y = y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total += y.size(0)
        total_train_loss += loss.item() * x.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == y).sum().item()
        
    train_loss = total_train_loss / total
    train_accuracy = correct / total

    # Validation
    val_loss, val_accuracy = evaluate(model, mnist_val_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    # Epoch summary
    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # Early stopping
    if early_stopping.step(model, val_loss, val_accuracy, epoch):
        print("\nEarly stopping triggered.")
        break


# Restore best model
early_stopping.restore(model)

print("\n================================================")
print("BEST MODEL SUMMARY")
print("================================================")
print(f"Best Epoch         : {early_stopping.best_epoch}")
print(f"Best Validation Acc: {early_stopping.best_acc:.4f}")
print()

# Training Curves

We visualize:
- training loss,
- validation loss,
- training accuracy,
- and validation accuracy

across epochs.

These curves help diagnose:
- convergence behavior,
- underfitting,
- overfitting,
- and optimization stability.

The highlighted validation minimum corresponds to the model checkpoint retained by early stopping.

In [ ]:
epochs_range = range(1, len(train_losses) + 1)

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

# Plot training and validation curves with distinct semantic colors and weights
plt.plot(epochs_range, train_losses, label="Training Loss", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_losses, label="Validation Loss", color="#d95f02", linewidth=2.5, linestyle="--")

# Highlight the minimum validation loss point (where Early Stopping saves weights)
best_epoch = early_stopping.best_epoch
best_loss = early_stopping.best_loss
plt.scatter(best_epoch, best_loss, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

# Enhancing text and metadata
plt.title("Training vs Validation Loss", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Cross Entropy Loss", fontsize=11, labelpad=10)

# Refined grid lines for easy readability without visual clutter
plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
epochs_range = range(1, len(train_losses) + 1)
best_epoch = early_stopping.best_epoch
best_acc = early_stopping.best_acc

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

# Plot training and validation curves with distinct semantic colors and weights
plt.plot(epochs_range, train_accuracies, label="Training Accuracy", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_accuracies, label="Validation Accuracy", color="#d95f02", linewidth=2.5, linestyle="--")

# Highlight the minimum validation loss point (where Early Stopping saves weights)
plt.scatter(best_epoch, best_acc, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

# Enhancing text and metadata
plt.title("Training vs Validation Accuracy", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Accuracy", fontsize=11, labelpad=10)

# Refined grid lines for easy readability without visual clutter
plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

# Strategic legend placement
plt.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


# Final Test Evaluation

After training is complete, the best validation checkpoint is evaluated on the held-out test set.

The test set provides the most reliable estimate of real-world generalization performance because it was never used during:
- optimization,
- hyperparameter tuning,
- or model selection.

We compute:
- accuracy,
- macro F1 score,
- and weighted F1 score.

While accuracy measures overall correctness, F1 scores provide a more balanced evaluation of per-class performance.


In [ ]:
model.eval()

# 1. Initialize clean lists
all_predictions = []
all_labels = []

with torch.no_grad():
    for x, y in mnist_test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        predictions = logits.argmax(dim=1)

        # 2. CRITICAL: Append raw tensors directly. 
        # Do NOT use .extend() and do NOT call .cpu().numpy() here.
        all_predictions.append(predictions)
        all_labels.append(y)

# 3. Concatenate the tensors first, THEN move to CPU and convert to NumPy
all_predictions = torch.cat(all_predictions).cpu().numpy()
all_labels = torch.cat(all_labels).cpu().numpy()

accuracy = np.mean(all_predictions == all_labels)

f1_macro = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

f1_weighted = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print("\n================================================")
print("FINAL TEST METRICS")
print("================================================")

print(f"Accuracy      : {accuracy:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")
print(f"Weighted F1   : {f1_weighted:.4f}")


# Confusion Matrix

The confusion matrix visualizes:
- correct predictions,
- and systematic classification errors.

This analysis helps identify visually ambiguous digit pairs such as:
- 1 vs 7,
- 3 vs 8,
- and 6 vs 9.

Examining misclassification structure often provides more insight than aggregate metrics alone.


In [ ]:
class_names = [str(i) for i in range(10)]

cm = confusion_matrix(
    all_labels,
    all_predictions
)

fig, ax = plt.subplots(figsize=(6, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False
)

ax.grid(False)

ax.set_xticklabels(class_names, rotation=0)

plt.title("MNIST Digits Confusion Matrix")
plt.show()


# Classification Report

The classification report summarizes:
- precision,
- recall,
- and F1 score

for each class individually.

This provides a more detailed view of model behavior and helps identify:
- difficult classes,
- asymmetric errors,
- and class-specific weaknesses.


In [ ]:
print("\n================================================")
print("CLASSIFICATION REPORT")
print("================================================")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
))

# Saving Pretrained Features

After training, the learned convolutional feature extractor is saved to disk.

Only the feature extraction layers are stored because they will later be reused for transfer learning experiments.


In [ ]:
#torch.save(model.state_dict(), "mnist_pretrained.pth")
torch.save(model.features.state_dict(), "mnist_features.pth")

print("\nSaved pretrained MNIST weights.")


# Conclusion

This notebook demonstrates a complete deep learning workflow for image classification using convolutional neural networks.

Through the MNIST digit recognition task, we explored several foundational ideas in computer vision and neural network training:

* convolutional feature extraction,
* mini-batch optimization,
* regularization,
* validation-based model selection,
* and classification evaluation.

The experiment also highlights several practical machine learning principles:

* preprocessing and normalization improve optimization stability,
* larger neural networks require regularization,
* validation performance is more important than training performance,
* and early stopping helps prevent overfitting.

By separating the model into:

* a feature extractor,
* and a classifier,

the notebook also introduces the foundations of transfer learning, where pretrained visual representations can be reused across tasks.

More broadly, this experiment illustrates one of the central ideas of deep learning:

> Neural networks can automatically learn increasingly useful hierarchical representations directly from raw data when trained with proper optimization, regularization, and evaluation procedures.
